# 1. Load S3 data and create model features

We use 15-minute bars from one year of BTCUSDT candles stored in S3. The target is DOWN, FLAT, or UP 60 minutes later. FLAT means the 60-minute move is smaller than 0.20% of the current price.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "s3_to_dataframe.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from s3_to_dataframe import load_data_from_s3, prepare_dataframe
from feature_engineering import FEATURE_COLUMNS, aggregate_candles, create_features

In [2]:
# Download the same CSV that is stored in S3 and keep a local copy for inspection.
raw_df = prepare_dataframe(load_data_from_s3())
data_dir = ROOT / "data"
data_dir.mkdir(exist_ok=True)
raw_df.to_csv(data_dir / "historical_btcusdt_15m_1y.csv", index=False)
print(f"Raw rows: {len(raw_df):,}")
print(f"Saved local view: {data_dir / 'historical_btcusdt_15m_1y.csv'}")
display(raw_df.head())

Raw rows: 35,004
Saved local view: c:\Users\aashu\OneDrive\Desktop\projects\dtsc3601-crypto-project\data\historical_btcusdt_15m_1y.csv


,open_time,open_time_utc,open,high,low,close,volume,close_time,close_time_utc,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,1758929400000,2025-09-26 23:30:00+00:00,109629.75,109715.63,109629.75,109662.00,0.05531,1758930299999,2025-09-26 23:44:59.999000+00:00,6066.922932,16,0.03371,3697.895316
1,1758930300000,2025-09-26 23:45:00+00:00,109715.63,109725.04,109715.63,109725.04,0.00338,1758931199999,2025-09-26 23:59:59.999000+00:00,370.840336,4,0.00338,370.840336
2,1758931200000,2025-09-27 00:00:00+00:00,109723.92,109723.92,109465.05,109705.58,0.00915,1758932099999,2025-09-27 00:14:59.999000+00:00,1003.384399,8,0.00084,92.160526
3,1758932100000,2025-09-27 00:15:00+00:00,109705.58,109725.00,109474.25,109715.95,0.02628,1758932999999,2025-09-27 00:29:59.999000+00:00,2883.081664,12,0.02617,2871.037009
4,1758933000000,2025-09-27 00:30:00+00:00,109718.20,109718.20,109394.67,109690.00,0.00404,1758933899999,2025-09-27 00:44:59.999000+00:00,443.149163,10,0.00392,430.020967


In [3]:
bars = aggregate_candles(raw_df, timeframe="15min")
feature_df = create_features(raw_df, timeframe="15min", horizon_bars=4, flat_threshold=0.002)
print(f"15-minute bars: {len(bars):,}")
print(f"Usable feature rows: {len(feature_df):,}")
print(f"Feature columns: {FEATURE_COLUMNS}")
display(feature_df[["open_time_utc", "close", *FEATURE_COLUMNS, "target"]].tail())

15-minute bars: 35,004
Usable feature rows: 34,094
Feature columns: ['return_1', 'return_3', 'return_6', 'return_12', 'ema_ratio', 'rsi_14', 'volatility_12', 'volume_ratio', 'range_pct', 'close_position', 'buy_volume_ratio', 'hour_sin', 'hour_cos', 'body_pct', 'upper_wick_pct', 'lower_wick_pct']


,open_time_utc,close,return_1,return_3,return_6,return_12,ema_ratio,rsi_14,volatility_12,volume_ratio,range_pct,close_position,buy_volume_ratio,hour_sin,hour_cos,body_pct,upper_wick_pct,lower_wick_pct,target
34089,2026-09-26 21:15:00+00:00,84200.00,0.000986,0.003641,0.002194,0.001566,0.000290,54.770529,0.000975,2.543765,0.001242,1.000000,0.662292,-0.659346,0.751840,0.000909,0.000000,0.000333,0
34090,2026-09-26 21:30:00+00:00,84158.37,-0.000494,0.001520,0.001499,0.001570,0.000414,53.623430,0.000975,1.100011,0.001024,0.000000,0.222157,-0.608761,0.793353,-0.000747,0.000277,0.000000,0
34091,2026-09-26 21:45:00+00:00,84157.60,-0.000009,0.000482,0.000838,0.001700,0.000479,56.543927,0.000972,1.330558,0.000744,0.620591,0.946030,-0.555570,0.831470,-0.000020,0.000262,0.000462,0
34092,2026-09-26 22:00:00+00:00,84322.28,0.001957,0.001452,0.005099,0.003557,0.000859,68.384555,0.001104,2.203347,0.002994,0.640195,0.198715,-0.500000,0.866025,0.001917,0.001077,0.000000,0
34093,2026-09-26 22:15:00+00:00,84357.68,0.000420,0.002368,0.003892,0.003979,0.001152,70.692964,0.001100,1.012625,0.001228,0.979145,0.658030,-0.442289,0.896873,0.000227,0.000026,0.000975,0


Features use only the current bar and trailing history: multi-window returns, EMA trend, RSI, rolling volatility, relative volume, candle range/position, taker-buy volume, and cyclical time-of-day. The future close is used only to create `target`, never as an input feature.

In [4]:
display(feature_df["target"].value_counts(normalize=True).rename({-1: "DOWN", 0: "FLAT", 1: "UP"}))
feature_df.to_csv(data_dir / "features_btcusdt_15min_60min.csv", index=False)
print(f"Saved feature view: {data_dir / 'features_btcusdt_15min_60min.csv'}")

target
FLAT    0.485511
UP      0.258081
DOWN    0.256409
Name: proportion, dtype: float64

Saved feature view: c:\Users\aashu\OneDrive\Desktop\projects\dtsc3601-crypto-project\data\features_btcusdt_15min_60min.csv
